# Decision Tree

## 1. Import Libraries

In [9]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [10]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# 2. HÀM ĐÁNH GIÁ (Lưu kết quả chung vào list)
results_list = []

def evaluate_model(model, version_name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    res = {
        'Version': version_name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_prob), 4)
    }
    
    results_list.append(res)
    print(f"[{version_name}] Acc: {res['Accuracy']} | F1: {res['F1-Score']} | AUC: {res['AUC']}")
    return model

###  Danh sách lưu kết quả để xuất CSV

In [11]:
results_list = []

def evaluate_model(model, name, X_test, y_test):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)"""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Lấy xác suất của class 1 để tính AUC
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    res = {
        'Algorithm': name,
        'Accuracy': round(acc, 2),
        'Precision': round(prec, 2),
        'Recall': round(rec, 2),
        'F1-Score': round(f1, 2),
        'AUC': round(auc, 2)
    }
    
    results_list.append(res)
    print(f"[{name}] Acc: {acc:.2f} | Prec: {prec:.2f} | Rec: {rec:.2f} | F1: {f1:.2f} | AUC: {auc:.2f}")
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [12]:
print("--- V1: Baseline Decision Tree ---")

dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train, y_train)

evaluate_model(dt_baseline, "V1: DT Baseline", X_test, y_test)

--- V1: Baseline Decision Tree ---
[V1: DT Baseline] Acc: 1.00 | Prec: 0.99 | Rec: 1.00 | F1: 1.00 | AUC: 1.00


{'Algorithm': 'V1: DT Baseline',
 'Accuracy': 1.0,
 'Precision': 0.99,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [13]:
print("--- V2: GridSearchCV Tuning Decision Tree ---")

param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 3, 5, 7, 10, 15],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
dt_tuned = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")
evaluate_model(dt_tuned, "V2: DT Tuned (GridSearch)", X_test, y_test)

--- V2: GridSearchCV Tuning Decision Tree ---
Best Params found by CV: {'class_weight': None, 'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
[V2: DT Tuned (GridSearch)] Acc: 0.98 | Prec: 0.98 | Rec: 0.98 | F1: 0.98 | AUC: 0.98


{'Algorithm': 'V2: DT Tuned (GridSearch)',
 'Accuracy': 0.98,
 'Precision': 0.98,
 'Recall': 0.98,
 'F1-Score': 0.98,
 'AUC': 0.98}

In [14]:
print("--- V3: Ensemble Learning (Stacking) ---")

base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]

dt_meta = DecisionTreeClassifier(
    random_state=42,
    max_depth=5,
    min_samples_leaf=2
)

v3_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=dt_meta,
    cv=5,
    n_jobs=-1
)

v3_model.fit(X_train, y_train)
evaluate_model(v3_model, "V3: DT Stacking Ensemble", X_test, y_test)

--- V3: Ensemble Learning (Stacking) ---
[V3: DT Stacking Ensemble] Acc: 1.00 | Prec: 0.99 | Rec: 1.00 | F1: 1.00 | AUC: 1.00


{'Algorithm': 'V3: DT Stacking Ensemble',
 'Accuracy': 1.0,
 'Precision': 0.99,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

### (5) Lưu kết quả

In [15]:
# 5. LƯU KẾT QUẢ VÀ MODEL
save_path = '../experiment/DecisionTree/'
os.makedirs(save_path, exist_ok=True)

df_results = pd.DataFrame(results_list)
csv_output = os.path.join(save_path, 'decision_tree_results.csv')
df_results.to_csv(csv_output, index=False)

with open(os.path.join(save_path, 'dt_baseline.pkl'), 'wb') as f:
    pickle.dump(dt_baseline, f)

with open(os.path.join(save_path, 'dt_tuned.pkl'), 'wb') as f:
    pickle.dump(dt_tuned, f)

with open(os.path.join(save_path, 'dt_stacking.pkl'), 'wb') as f:
    pickle.dump(v3_model, f)

print("-" * 40)
print(f"✅ Đã lưu file CSV tại: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)

----------------------------------------
✅ Đã lưu file CSV tại: ../experiment/DecisionTree/decision_tree_results.csv
✅ Đã lưu models tại: ../experiment/DecisionTree/
----------------------------------------


,Algorithm,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: DT Baseline,1.00,0.99,1.00,1.00,1.00
1,V2: DT Tuned (GridSearch),0.98,0.98,0.98,0.98,0.98
2,V3: DT Stacking Ensemble,1.00,0.99,1.00,1.00,1.00


In [16]:
!jupyter nbconvert --to html decision_tree.ipynb

[NbConvertApp] Converting notebook decision_tree.ipynb to html
[NbConvertApp] Writing 306453 bytes to decision_tree.html
